# Desafio 1 — Ciência de Dados 1

**Aluno(a): Fabiana Rotella Campaner**

Rode cada célula e **escreva sua resposta** no campo indicado. Você **não** precisa escrever código — o objetivo é interpretar e calcular a partir dos resultados. Os arquivos .csv estão nesta pasta.

## Exercício 1 — Log loss: o preço da confiança errada

Dois modelos (A e B) preveem os mesmos 5 exemplos.

**(a)** Os dois têm a MESMA acurácia. Por que o **log loss** difere?

**(b)** Qual modelo é preferível e por quê?

**(c)** Olhando a **perda por exemplo** de B, qual exemplo mais pesa no log loss? Explique com o valor.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import log_loss, accuracy_score
df=pd.read_csv("ex4_modelos.csv"); y=df["y"].values; pA=df["pA"].values; pB=df["pB"].values
predA=(pA>=0.5).astype(int); predB=(pB>=0.5).astype(int)
print("acuracia A:",accuracy_score(y,predA)," | acuracia B:",accuracy_score(y,predB))
print("log loss A:",round(log_loss(y,pA),3)," | log loss B:",round(log_loss(y,pB),3))
# perda de cada exemplo do modelo B:  -log(prob. dada a classe CERTA)
perdaB = -np.log(np.where(y==1, pB, 1-pB))
for i,(yy,pp,perda) in enumerate(zip(y,pB,perdaB)): print(f"ex{i}: y={yy} pB={pp} perda={perda:.2f}")
p=np.linspace(0.01,1,200); plt.plot(p,-np.log(p)); plt.xlabel("prob. dada a classe CERTA"); plt.ylabel("-log(p)"); plt.title("Log loss de um exemplo"); plt.tight_layout(); plt.show()

**Resposta:**

_(escreva aqui)_

## Exercício 2 — Limiar, matriz de confusão e ROC

O gráfico mostra a nota de 15 emails (cor = spam real ou não) com o limiar em **0,5** (spam = positivo).

**(a)** Com o limiar 0,5, defina VP, VN, FP e FN.

**(b)** O ideal é o **maior recall possível**. O que fazer com o limiar? Explique.

**(c)** Avalie a **ROC/AUC**: o que ela representa e o que faria a curva melhorar?

**(d)** Agora suba o limiar para **0,7**: dê os **novos VP, FP e FN** e **recalcule precisão e recall**. Compare com o de 0,5.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as mp
from sklearn.metrics import roc_curve, auc
df=pd.read_csv("ex3_emails.csv"); nota=df["nota"].values; spam=df["spam"].values; m=len(df)
lim=0.5; ordem=np.argsort(nota)
plt.figure(figsize=(8,6))
cores=["#B91C1C" if s==1 else "#0F766E" for s in spam[ordem]]
plt.barh(range(m), nota[ordem], color=cores); plt.axvline(lim, ls="--", color="black")
plt.yticks(range(m), [f"email {i}" for i in ordem])
for i,v in enumerate(nota[ordem]): plt.text(v+0.01,i,str(v),va="center",fontsize=8)
plt.legend(handles=[mp.Patch(color="#B91C1C",label="spam (real)"),mp.Patch(color="#0F766E",label="nao-spam (real)"),
                    plt.Line2D([0],[0],ls="--",color="black",label="limiar = 0.5")], loc="lower right")
plt.xlabel("nota (prob. de ser spam)"); plt.title("Notas por email"); plt.tight_layout(); plt.show()
fpr,tpr,_=roc_curve(spam,nota); a=auc(fpr,tpr)
plt.figure(figsize=(4.5,4)); plt.plot(fpr,tpr); plt.plot([0,1],[0,1],"--",color="gray"); plt.title(f"ROC (AUC={a:.2f})"); plt.xlabel("FPR"); plt.ylabel("TPR"); plt.tight_layout(); plt.show()

**Resposta:**

_(escreva aqui)_

## Exercício 3 — Acurácia alta = modelo bom?

O código dá VP, VN, FP, FN (limiar 0,5), a proporção de positivos e as curvas ROC e PR.

**(a)** Calcule **acurácia**, **recall** e **precisão** a partir de VP, VN, FP e FN.

**(b)** O desenvolvedor disse que o modelo é bom **porque a acurácia é alta**. Ele está certo? Justifique usando a **proporção de positivos** e o que a ROC e a PR mostram.

**(c)** Se o modelo passasse a acertar **2 dos FN** (viram VP), qual seria o **novo recall**? Mostre a conta.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve
df = pd.read_csv("ex2_classificacao.csv"); y=df["y_real"].values; score=df["score"].values
pred=(score>=0.5).astype(int)
VP=int(((pred==1)&(y==1)).sum()); FP=int(((pred==1)&(y==0)).sum())
FN=int(((pred==0)&(y==1)).sum()); VN=int(((pred==0)&(y==0)).sum())
print("VP =",VP,"| FP =",FP,"| FN =",FN,"| VN =",VN,"| N =",len(y))
print("proporcao de positivos (reais):", round(y.mean(),3))
fpr,tpr,_=roc_curve(y,score); prec,rec,_=precision_recall_curve(y,score)
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(fpr,tpr); ax[0].plot([0,1],[0,1],"--",color="gray"); ax[0].set_title(f"ROC (AUC={auc(fpr,tpr):.2f})"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR")
ax[1].plot(rec,prec); ax[1].axhline(y.mean(),ls="--",color="red",label=f"base = {y.mean():.2f}"); ax[1].set_title("Precisao-Revocacao"); ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precisao"); ax[1].legend()
plt.tight_layout(); plt.show()

**Resposta:**

_(escreva aqui)_

## Exercício 4 — Overfitting × underfitting

A acurácia de treino e teste por profundidade da árvore.

**(a)** Aponte a região de **underfitting** e a de **overfitting**.

**(b)** O que acontece com treino e teste em cada região?

**(c)** Qual complexidade você escolheria e por quê?

**(d)** A partir de qual profundidade o **vão treino−teste** começa a se abrir, e o que isso indica?

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
df=pd.read_csv("ex6_dados.csv"); X=df[[f"f{i}" for i in range(1,7)]]; y=df["y"]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,random_state=0,stratify=y)
prof=range(1,16); tr=[]; te=[]
for d in prof:
    m=DecisionTreeClassifier(max_depth=d,random_state=0).fit(Xtr,ytr)
    tr.append(accuracy_score(ytr,m.predict(Xtr))); te.append(accuracy_score(yte,m.predict(Xte)))
plt.plot(list(prof),tr,marker="o",label="treino"); plt.plot(list(prof),te,marker="o",label="teste")
plt.xlabel("profundidade (complexidade)"); plt.ylabel("acuracia"); plt.legend(); plt.title("Treino x teste"); plt.tight_layout(); plt.show()
print("melhor profundidade no teste:", list(prof)[int(np.argmax(te))])

**Resposta:**

_(escreva aqui)_

## Exercício 5 — MAE, RMSE, R² e MAPE: qual usar?

MAE, RMSE, R² e MAPE para duas colunas de escalas diferentes.

**(a)** Na Coluna A, por que MAE e RMSE ficam tão diferentes?

**(b)** Dá para comparar o MAE da Coluna A com o da Coluna B para dizer onde o modelo é melhor? Por quê?

**(c)** Por que o MAPE da Coluna B fica tão alto?

**(d)** Qual medida é a mais adequada para julgar o modelo? Justifique.

**(e)** Se o MAIOR erro da Coluna A **dobrasse**, qual muda mais: **MAE** ou **RMSE**? Por quê?

In [ ]:
import pandas as pd, numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
df=pd.read_csv("ex5_regressao.csv")
def metr(yr,yp):
    return (mean_absolute_error(yr,yp), mean_squared_error(yr,yp)**0.5, r2_score(yr,yp), np.mean(np.abs((yr-yp)/yr))*100)
for nome,cr,cp in [("Coluna A (grande, com outlier)","colA_real","colA_prev"),("Coluna B (pequena, perto de zero)","colB_real","colB_prev")]:
    mae,rmse,r2,mape=metr(df[cr],df[cp]); print(nome); print(f"   MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.3f}  MAPE={mape:.1f}%")

**Resposta:**

_(escreva aqui)_

## Exercício 6 — Dois modelos e o custo do erro

Um modelo detecta **falha** em um equipamento. Um **FN** (falha não detectada) causa uma parada cara; um **FP** (alarme falso) é barato (só uma checagem). O gráfico mostra a matriz de confusão de dois modelos, A e B.

**(a)** Calcule o **recall** e a **precisão** de A e de B.

**(b)** Qual modelo você escolhe para ESTE caso? Justifique pelo **custo dos erros**.

**(c)** O que você faria com o **limiar** do modelo escolhido para melhorar ainda mais o que importa aqui? Explique o efeito em recall e precisão.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
df = pd.read_csv("ex1_modelos.csv"); print(df.to_string(index=False))
cats=["VP","FP","FN"]; x=np.arange(3)   # os que importam para recall e precisao
A=df[df.modelo=="A"][cats].values[0]; B=df[df.modelo=="B"][cats].values[0]
plt.bar(x-0.2,A,0.4,label="Modelo A"); plt.bar(x+0.2,B,0.4,label="Modelo B")
for xi,(a,b) in enumerate(zip(A,B)):
    plt.text(xi-0.2,a+0.5,str(a),ha="center",fontsize=9); plt.text(xi+0.2,b+0.5,str(b),ha="center",fontsize=9)
plt.xticks(x,cats); plt.ylabel("nº de casos"); plt.title("Matriz de confusão (VP, FP, FN) — A x B"); plt.legend(); plt.show()

**Resposta:**

_(escreva aqui)_